# Plot maps training data

In [ ]:
# Libraries
import numpy as np
import xarray as xr
import dask.dataframe as dd
import geopandas as gpd
from shapely.geometry import box
import matplotlib.pyplot as plt
import matplotlib.colors as colors

In [ ]:
# Directories
dir01 = '../paper_ecosystem_carbon/output/01_data_prep/'
dir02 = '../paper_ecosystem_carbon/output/02_database_prep/'
dir07 = '../paper_ecosystem_carbon/output/07_eval_prep/'
dir08 = '../paper_ecosystem_carbon/output/08_eval/'
dir_nearth = '../data/naturalearth/'

---

### Pre-processing

In [ ]:
# Initialize dask
from dask_jobqueue import SLURMCluster
cluster = SLURMCluster(queue='interactive', 
                       cores=12, #6
                       memory='256 GB',
                       account='bm0891', 
                       interface='ib0',
                       walltime='02:00:00',
                       local_directory='../dask/',
                       job_extra_directives=[
                           '-o ../dask/LOG_worker_%j.o',
                           '-e ../dask/LOG_worker_%j.o'])

# Scale dask clusters
cluster.scale(jobs=1)

# Configurate dashboard url
import dask
dask.config.config.get('distributed').get('dashboard') \
    .update({'link':'{JUPYTERHUB_SERVICE_PREFIX}/proxy/{port}/status'})

# Create client
from dask.distributed import Client
client = Client(cluster)

client

In [ ]:
# Get training data 
dd_base = dd.read_parquet(f'{dir02}dd_base_r001.parquet')
# Prepared occurance data
ds_occur = xr.open_zarr(f'{dir07}ds_prep_occur_r025.zarr')
# Prepared dinerstein data
ds_din = xr.open_zarr(f'{dir01}ds_prep_dinerstein_etype_r025.zarr')
# Get coastline data
coastline110 = gpd.read_file(
    f'{dir_nearth}ne_110m_coastline/ne_110m_coastline.shp')

In [ ]:
# List with ecosystem types
list_etype = ['trop_ef', 'trop_df', 'temp_ebf', 'temp_dbf', 'temp_enf',
              'temp_dnf', 'glob_shrub', 'glob_grass', 'glob_tundr', 
              'glob_mount']

# Dictionary with ecosystem types and original names
dict_etype = dict(
    trop_ef = ['Tropical/subtropical evergreen forest', 
               'Tropical/subtropical\nevergreen forest'], 
    trop_df = ['Tropical/subtropical deciduous forest', 
               'Tropical/subtropical\ndeciduous forest'],
    temp_ebf = ['Temperate/boreal evergreen broadleaf forest', 
                'Temperate/boreal\nevergreen broadleaf forest'],
    temp_dbf = ['Temperate/boreal deciduous broadleaf forest', 
                'Temperate/boreal\ndeciduous broadleaf forest'],
    temp_enf = ['Temperate/boreal evergreen needleleaf forest', 
                'Temperate/boreal\nevergreen needleleaf forest'],
    temp_dnf = ['Temperate/boreal deciduous needleleaf forest', 
                'Temperate/boreal\ndeciduous needleleaf forest'],
    glob_shrub = ['Shrublands', 'Shrublands'],
    glob_grass = ['Grasslands','Grasslands'],
    glob_tundr = ['Tundra', 'Tundra'],
    glob_mount = ['Montane', 'Montane'])

In [ ]:
# Create subset of dd_base with relevant columns and load to memory
dd_base_sel = dd_base[
    ['lat', 'lon', *[i for i in dd_base.columns if i.startswith('train_')]]
    ].persist()

---

### Plot functions

In [ ]:
def get_grid_train(ctype, etype, tile_size):

    """Creates dataframe with count of trianing data within the defined 
    tile_size°"""
    
    # Define training data column
    train_col = f'train_{ctype}_{etype}'
    
    # Get training data
    dd_sel = dd_base_sel[dd_base_sel[train_col] == True] \
        [['lat', 'lon', train_col]].persist()
    
    # Create geodataframe
    geometry = gpd.points_from_xy(dd_sel.lon, dd_sel.lat)
    gdf = gpd.GeoDataFrame(dd_sel, geometry=geometry, crs='EPSG:4326')
    
    # Get the geographical bounds of the data
    minx, miny, maxx, maxy = gdf.total_bounds
        
    # Generate a grid with size tile_size covering the extent of the data
    grid_cells = []
    for x in np.arange(np.floor(minx), np.ceil(maxx), tile_size):
        for y in np.arange(np.floor(miny), np.ceil(maxy), tile_size):
            grid_cells.append(box(x, y, x + tile_size, y + tile_size))
    
    # Create a GeoDataFrame for the grid
    grid = gpd.GeoDataFrame(grid_cells, columns=['geometry'], crs='EPSG:4326')
    
    # Perform a spatial join to assign each point to a grid cell
    joined = gpd.sjoin(gdf, grid, how='left', predicate='within')
    
    # Count the number of points in each grid cell
    grid['count'] = joined.groupby('index_right').size()

    return grid

In [ ]:
def plot_sub(ax, ctype, etype, cmap, vmax, tile_size, add_legend=False):

    """Plots training data of one ecosystem type"""
    
    # Create geodataframe with count of training data per 1*1 degree grid tile 
    grid = get_grid_train(ctype, etype, tile_size)

    # Dataset with occurance of etype
    ds_occur[etype].plot.imshow(ax=ax, cmap='Greys', vmin=0, vmax=2, add_colorbar=False, alpha=0.5)
    
    # Plot training grid cell count
    grid.plot(ax=ax, column='count', cmap=cmap, vmin=0, vmax=vmax, legend=False)
    
    # Plot coastline
    coastline110.plot(ax=ax, color='#000000', linewidth=0.5)
    
    ax.axis('off')
    
    # Set title
    ax.set_title(f'{dict_etype[etype][0]}\nn = {int(grid['count'].sum())}')

    # Limit y axis extent
    ax.set_ylim(-60, 85)

    if add_legend==True:
        # Add a colorbar to indicate the density of points in grid cells
        norm = colors.Normalize(vmin=0, vmax=vmax)
        cbar = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
        
        # Ensure the colorbar is tied to the figure containing the axes
        fig = ax.get_figure()
        ax_cbar = fig.colorbar(cbar, ax=ax, shrink=0.9)
        ax_cbar.set_label(f'Training points\nper {tile_size}°x{tile_size}° grid tile')

---

### Plot

In [ ]:
%%time
# Define parameters
cmap = 'Spectral_r'
vmax = 250
tile_size = 2.5

# Plot
fig, axs = plt.subplots(figsize=(10, 12), ncols=2, nrows=5, dpi=300)
axs = axs.ravel()

# Loop through ecosystem types and plot training data
for i in range(0, len(list_etype)):
    plot_sub(axs[i], 'cveg', list_etype[i], cmap, vmax, tile_size)

# Colorbar
norm = colors.Normalize(vmin=0, vmax=vmax)
cbar = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
cbar_ax = fig.add_axes([0.2, 0.05, 0.6, 0.01])  # [left, bottom, width, height]
fig.colorbar(cbar, cax=cbar_ax, orientation='horizontal',
             shrink=0.6, aspect=70, pad=0.1, 
             label=f'Training points by {tile_size}°x{tile_size}° grid tile')

# Layout
plt.subplots_adjust(hspace=0.5, bottom=0.08)

# Export
fstr_out = 'fig_s01_training data'
plt.savefig(f'{dir08}pdf/{fstr_out}.pdf', dpi=300, 
            bbox_inches='tight', pad_inches=0.1); 
plt.savefig(f'{dir08}png/{fstr_out}.png', dpi=300, 
            bbox_inches='tight', pad_inches=0.1);

In [ ]:
cluster.close()